# Survey Results

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum

import numpy as np
import pandas as pd
from IPython.display import Markdown

pd.set_option('future.no_silent_downcasting', True)

# Raw Data

In [2]:
responses = pd.read_csv("survey_responses.csv")
responses.head()

,Timestamp,Username,Rank options [implicit-builder],Rank options [explicit],Rank options [implicit-only-string],Rank options [implicit-no-builder],Rank options [extensible-th],Rank options [extensible-hasclass],Which options do you like? [implicit-builder],Which options do you like? [explicit],Which options do you like? [implicit-only-string],Which options do you like? [implicit-no-builder],Which options do you like? [extensible-th],Which options do you like? [extensible-hasclass],Comments or Questions?
0,2025-04-21 22:55:11,<scrubbed>,5.0,6.0,2.0,3.0,4.0,1.0,Somewhat happy,Really unhappy,Somewhat happy,Somewhat happy,Somewhat happy,Really happy,<scrubbed>
1,2025-04-21 23:06:23,<scrubbed>,2.0,5.0,6.0,4.0,1.0,3.0,Really happy,Somewhat unhappy,Really unhappy,Really unhappy,Really happy,Somewhat unhappy,<scrubbed>
2,2025-04-21 23:07:57,<scrubbed>,1.0,6.0,5.0,4.0,3.0,2.0,Really happy,Really unhappy,Somewhat unhappy,Somewhat unhappy,Ambivalent,Somewhat happy,<scrubbed>
3,2025-04-21 23:28:01,<scrubbed>,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,Somewhat happy,NaN,<scrubbed>
4,2025-04-21 23:48:09,<scrubbed>,3.0,5.0,6.0,2.0,1.0,4.0,Ambivalent,Somewhat unhappy,Really unhappy,Somewhat happy,Really happy,Somewhat happy,<scrubbed>


# Ranked-choice results

In [3]:
class InterpolationOption(Enum):
    IMPLICIT_BUILDER = "implicit-builder"
    EXPLICIT = "explicit"
    IMPLICIT_ONLY_STRING = "implicit-only-string"
    IMPLICIT_NO_BUILDER = "implicit-no-builder"
    EXTENSIBLE_TH = "extensible-th"
    EXTENSIBLE_HASCLASS = "extensible-hasclass"

@dataclass(frozen=True)
class Ballot:
    # ranking[0] is the top rank
    ranking: list[InterpolationOption]

    @classmethod
    def load(cls, data: pd.Series) -> Ballot:
        ranking_map = {
            int(rank): option
            for option in InterpolationOption.__members__.values()
            if not np.isnan(rank := data[f"Rank options [{option.value}]"])
        }
        ranking = [option for _, option in sorted(ranking_map.items())]
        return cls(ranking=ranking)

    def get_top(self) -> InterpolationOption | None:
        return self.ranking[0] if self.ranking else None

    def get_rank(self, option: InterpolationOption) -> int | None:
        try:
            return self.ranking.index(option)
        except ValueError:
            return None

    def get_next(self, option: InterpolationOption) -> InterpolationOption | None:
        i = self.get_rank(option)
        if i is not None and i + 1 < len(self.ranking):
            return self.ranking[i + 1]
        else:
            return None

@dataclass(frozen=True)
class Scoreboard:
    votes: dict[InterpolationOption, list[Ballot]]

    @classmethod
    def init(cls, ballots: list[Ballot]) -> Scoreboard:
        votes = {
            option: [ballot for ballot in ballots if ballot.get_top() == option]
            for option in InterpolationOption.__members__.values()
        }
        return cls(votes=votes)

    def get_tally(self) -> RoundTally:
        tally = {option: len(ballots) for option, ballots in self.votes.items()}
        return RoundTally(tally=tally, total=sum(tally.values()))

    def remove(self, loser: InterpolationOption) -> Scoreboard:
        new_votes = self.votes.copy()
        loser_ballots = new_votes.pop(loser)

        for ballot in loser_ballots:
            new_vote = loser
            while new_vote is not None and new_vote not in new_votes:
                new_vote = ballot.get_next(new_vote)
            if new_vote is not None:
                new_votes[new_vote].append(ballot)

        return Scoreboard(votes=new_votes)

@dataclass(frozen=True)
class RoundTally:
    tally: dict[InterpolationOption, int]
    total: int

    def to_pandas(self) -> pd.DataFrame:
        options, counts = zip(*self.tally.items())
        return (
            pd.DataFrame({"option": [o.value for o in options], "count": counts})
            .set_index("option")
            .sort_values("count", ascending=False)
        )

    def get_winner(self) -> InterpolationOption | None:
        for option, count in self.tally.items():
            if count > self.total / 2:
                return option
        return None

    def get_loser(self) -> InterpolationOption:
        return min(self.tally.items(), key=lambda pair: pair[1])[0]

In [4]:
ballots = [
    Ballot.load(data)
    for _, data in responses.iterrows()
]

scoreboard = Scoreboard.init(ballots)
for round_num in range(len(InterpolationOption.__members__)):
    display(Markdown(f"---\n**Round {round_num + 1}**"))
    round_tally = scoreboard.get_tally()
    display(round_tally.to_pandas())
    winner = round_tally.get_winner()
    if winner:
        display(Markdown(f"Winner: **{winner.value}**"))
        break

    loser = round_tally.get_loser()
    scoreboard = scoreboard.remove(loser)

---
**Round 1**

,count
option,
implicit-builder,24
extensible-th,19
explicit,14
implicit-no-builder,14
extensible-hasclass,8
implicit-only-string,7


---
**Round 2**

,count
option,
implicit-builder,26
extensible-th,19
explicit,16
implicit-no-builder,15
extensible-hasclass,10


---
**Round 3**

...

---
**Round 4**

,count
option,
implicit-builder,30
implicit-no-builder,29
extensible-th,27


---
**Round 5**

,count
option,
implicit-no-builder,42
implicit-builder,41


Winner: **implicit-no-builder**

`implicit-no-builder` really rocketed up at the end, indicating it's the option with the most widespread support. The results from the approval question also support this (see next section).

`implicit-builder` remained strong thoughout

Notably, `extensible-th` had a strong initial showing, but didn't gain much over the rounds, indicating strong support from a cohort, but low support from people who didn't have it as their top choice.

# Approval results

In addition to the ranked-choice question, I also asked responders to indicate their happiness with the various options. The aim was to distinguish between "relative ranking of options to each other" and "objective likability of the option, in isolation".

First, we'll look at the raw data:

In [5]:
approvals = pd.DataFrame({
    option.value: responses[f"Which options do you like? [{option.value}]"].replace(np.nan, "N/A")
    for option in InterpolationOption.__members__.values()
})
approvals

,implicit-builder,explicit,implicit-only-string,implicit-no-builder,extensible-th,extensible-hasclass
0,Somewhat happy,Really unhappy,Somewhat happy,Somewhat happy,Somewhat happy,Really happy
1,Really happy,Somewhat unhappy,Really unhappy,Really unhappy,Really happy,Somewhat unhappy
2,Really happy,Really unhappy,Somewhat unhappy,Somewhat unhappy,Ambivalent,Somewhat happy
3,N/A,N/A,N/A,N/A,Somewhat happy,N/A
4,Ambivalent,Somewhat unhappy,Really unhappy,Somewhat happy,Really happy,Somewhat happy
...,...,...,...,...,...,...
81,Somewhat happy,Ambivalent,Really unhappy,Somewhat happy,Somewhat happy,Ambivalent
82,Somewhat unhappy,Somewhat happy,Somewhat happy,Really happy,Really unhappy,Really unhappy
83,Ambivalent,Ambivalent,Ambivalent,Ambivalent,Somewhat unhappy,Somewhat happy
84,Really unhappy,Really unhappy,Really unhappy,Really unhappy,Somewhat unhappy,N/A


Counts of each option:

In [6]:
approval_counts = approvals.apply(lambda s: s.value_counts()).reindex([
    "Really unhappy",
    "Somewhat unhappy",
    "Ambivalent",
    "Somewhat happy",
    "Really happy",
    "N/A",
]).transpose().sort_values("Really happy", ascending=False)
approval_counts

,Really unhappy,Somewhat unhappy,Ambivalent,Somewhat happy,Really happy,N/A
implicit-builder,6,14,14,25,21,6
extensible-th,26,7,17,15,17,4
explicit,22,16,16,11,13,8
implicit-no-builder,9,12,19,29,12,5
extensible-hasclass,7,17,20,23,12,7
implicit-only-string,19,33,12,10,7,5


Grouped by general happiness level:

In [7]:
pd.DataFrame({
    "Unhappy": approval_counts["Really unhappy"] + approval_counts["Somewhat unhappy"],
    "Ambivalent": approval_counts["Ambivalent"] + approval_counts["N/A"],
    "Happy": approval_counts["Really happy"] + approval_counts["Somewhat happy"],
}).sort_values("Happy", ascending=False)

,Unhappy,Ambivalent,Happy
implicit-builder,20,20,46
implicit-no-builder,21,24,41
extensible-hasclass,24,27,35
extensible-th,33,21,32
explicit,38,24,24
implicit-only-string,52,17,17


Initial thoughts:
* Unsurprisingly, `extensible-th` is very controversial, with a 50-50 split and a very bi-modal distribution
    * Both `extensible-*` variants are fairly controversial
* "Unhappiness" is monotonically increasing as "Happy" goes down; the correlation is nice to see
* Surprisingly, people dislike `implicit-only-string` even more than `explicit`

# Commentary

I think `implicit-no-builder` is a good option, generally well-liked across the board. Most of the `implicit-builder` ballots ranked `implicit-no-builder` as their next choice too. Overall, I see a general trend of prioritizing simplicity over all else.

That being said, I do think the survey indicates a desire for _some_ extensibility, even if there's no great consensus on the exact mechanics of it. In the comments section, [@evincarofautumn](https://github.com/evincarofautumn) had a really interesting suggestion: taking a page from `QualifiedDo` and supporting syntax like `Mod."a ${x}"`. This person came at the suggestion from the perspective of taking syntax away from function application (which IMO is not an issue; is anyone applying functions to literals without spaces??), but I like it because it allows the mechanics to be customized **per-module**! The module could define a concretized `interpolate` function that shows much better type errors than a generalized extensible mechanism would do.

With both simplicity and extensibility in mind, I realized we might be able to eat our cake and have it too:
1. `-XStringInterpolation` would enable string interpolation with `implicit-no-builder`. Without `-XOverloadedStrings`, the overall string type is fixed at `String` (with interpolations using `Interpolate a String` instances), and with `-XOverloadedStrings`, the overall string type is any `IsString s`.
2. `-XQualifiedLiterals` would enable qualifying string literals (and, in the future, numeric literals too) and the qualified module should define `fromString`, `fromParts`, and `interpolate`, injected at the appropriate places:

```haskell
SQL."select * from users where name = ${name}"

SQL.fromParts
  [ SQL.fromString "select * from users where name = "
  , SQL.interpolate name
  ]
```

As a bonus, this provides benefit even outside of string interpolation; `text` could provide a module like

```haskell
module Data.Text.Overloaded where

fromString :: String -> Text
fromString = T.pack
```

and then the following code would be unambiguous:

```haskell
{-# LANGUAGE QualifiedLiterals #-}

import Data.Text.Overloaded qualified as T

-- printing "a" would've been ambiguous with OverloadedStrings
main = do
  print "string"
  print T."text"
```

So `-XStringInterpolation` gets the simple mechanism, with more complex mechanisms available for libraries to plug into. While these are conceivably two different proposals, I will propose them as a bundle in the proposal, as I think the string interpolation feature will get a lot more excitement alongside `QualifiedLiterals` than without. It's not without precedent (e.g. `OverloadedRecordDot`/`OverloadedRecordUpdate`), so I'm hoping we can ship the two together, but if needed, I could split into separate proposals.

# Appendix

## implicit-builder's next choice

In [8]:
# ballots ranking implicit-builder above implicit-no-builder
implicit_builder_ballots = [
    ballot
    for ballot in ballots
    if (
        (implicit_builder_rank := ballot.get_rank(InterpolationOption.IMPLICIT_BUILDER)) is not None
        and (implicit_no_builder_rank := ballot.get_rank(InterpolationOption.IMPLICIT_NO_BUILDER)) is not None
        and implicit_builder_rank < implicit_no_builder_rank
    )
]

# choice after implicit-builders
(
    pd.DataFrame([
        ballot.get_next(InterpolationOption.IMPLICIT_BUILDER).value
        for ballot in implicit_builder_ballots
    ])
    .value_counts()
)

0                   
implicit-no-builder     17
extensible-hasclass     14
explicit                 3
extensible-th            2
implicit-only-string     2
Name: count, dtype: int64